# IRIS-X ARC-AGI-2 Trial 001 — checkpointing offline Kaggle adapter v4
Run only after a separate explicit owner authorization for the exact run-002 package. Attach the private unpacked-model Dataset and the private Kaggle-extracted trial Dataset at `/kaggle/input/datasets/danielwuu/trial-input`, select one NVIDIA RTX PRO 6000 96 GB, and disable Internet. To resume a prior session, optionally attach a private Dataset containing `checkpoint-package.zip` at `/kaggle/input/datasets/danielwuu/run-002-checkpoint`.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, stat, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
TRIAL_DATASET_ROOT = INPUT / 'datasets' / 'danielwuu' / 'trial-input'
CHECKPOINT_DATASET_ROOT = INPUT / 'datasets' / 'danielwuu' / 'run-002-checkpoint'
MAX_FILES = 4097
MAX_FILE = 32 * 1024**3
MAX_TOTAL = 100 * 1024**3

def digest(data): return 'sha256:' + hashlib.sha256(data).hexdigest()
def file_digest(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024): hasher.update(chunk)
    return 'sha256:' + hasher.hexdigest()

def verify_and_copy_mounted_trial(source_root, target, expected_kind):
    if not source_root.is_dir(): raise RuntimeError(f'bound extracted trial Dataset is missing: {source_root}')
    all_paths = list(source_root.rglob('*'))
    if any(path.is_symlink() for path in all_paths): raise RuntimeError('trial Dataset contains a symbolic link')
    files = {path.relative_to(source_root).as_posix(): path for path in all_paths if path.is_file()}
    if len(files) > MAX_FILES or 'manifest.json' not in files: raise RuntimeError('trial Dataset file-count or manifest requirement failed')
    manifest_raw = files['manifest.json'].read_bytes()
    manifest = json.loads(manifest_raw)
    canonical = (json.dumps(manifest, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8')
    if manifest_raw != canonical or manifest.get('format') != 'irisx/closed-world-zip' or manifest.get('format_version') != '1.0': raise RuntimeError('non-canonical or unsupported manifest')
    if manifest.get('package_kind') != expected_kind: raise RuntimeError('package kind mismatch')
    declared = {item['path']: item for item in manifest.get('files', [])}
    if len(declared) != len(manifest.get('files', [])) or set(declared) != set(files) - {'manifest.json'}: raise RuntimeError('closed-world membership mismatch')
    if target.exists(): shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=False)
    total = 0
    for name, entry in declared.items():
        source = files[name]; size = source.stat().st_size; total += size
        if size > MAX_FILE or total > MAX_TOTAL: raise RuntimeError(f'trial Dataset byte limit exceeded: {name}')
        destination = target.joinpath(*name.split('/')); destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)
        if size != entry['byte_size'] or file_digest(destination) != entry['content_digest']: raise RuntimeError(f'file binding mismatch: {name}')
    (target / 'manifest.json').write_bytes(manifest_raw)
    return manifest, digest(manifest_raw)

trial_manifest, trial_manifest_digest = verify_and_copy_mounted_trial(TRIAL_DATASET_ROOT, WORKING / 'irisx-trial', 'irisx/arc-agi-2-trial-input')
if trial_manifest.get('package_id') != 'run-002': raise RuntimeError('v4 notebook requires the run-002 outbound package')
if file_digest(WORKING / 'irisx-trial/runtime/runner.py') != trial_manifest['metadata']['runner_digest']: raise RuntimeError('runner digest mismatch')
if file_digest(WORKING / 'irisx-trial/runtime/runner_core.py') != trial_manifest['metadata']['runner_core_digest']: raise RuntimeError('runner core digest mismatch')
if file_digest(WORKING / 'irisx-trial/runtime/notebook.ipynb') != trial_manifest['metadata']['notebook_digest']: raise RuntimeError('notebook digest mismatch')
provenance_path = WORKING / 'irisx-trial/bindings/model-source-provenance.json'
provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
unsigned = {key: provenance[key] for key in ('schema', 'schema_version', 'artifact_id', 'version', 'payload')}
if provenance.get('schema') != 'irisx/model-source-provenance' or provenance.get('content_digest') != digest(json.dumps(unsigned, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(',', ':')).encode('utf-8')): raise RuntimeError('model provenance artifact is invalid')
if provenance['content_digest'] != trial_manifest['metadata']['model']['source_provenance_digest']: raise RuntimeError('model provenance binding mismatch')
wheel_manifest_path = WORKING / 'irisx-trial/runtime/wheels.json'
wheel_manifest = json.loads(wheel_manifest_path.read_text(encoding='utf-8'))
if wheel_manifest.get('format') != 'irisx/offline-python-wheel-closure' or wheel_manifest.get('python_abi') != 'cp312' or wheel_manifest.get('platform') != 'manylinux2014_x86_64' or wheel_manifest.get('transformers') != '5.8.1': raise RuntimeError('unsupported offline runtime wheel closure')
if sys.version_info[:2] != (3, 12) or platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'): raise RuntimeError('offline runtime wheels require Linux x86_64 and Python 3.12')
runtime_site = WORKING / 'irisx-runtime-site'
if runtime_site.exists(): shutil.rmtree(runtime_site)
runtime_site.mkdir(parents=True, exist_ok=False)
extracted = set()
for entry in wheel_manifest.get('wheels', []):
    logical = entry.get('path', ''); wheel = WORKING / 'irisx-trial' / logical
    if not logical.startswith('runtime/wheels/') or wheel.name != logical.rsplit('/', 1)[-1] or not wheel.is_file(): raise RuntimeError(f'unsafe or missing runtime wheel: {logical}')
    if wheel.stat().st_size != entry.get('byte_size') or file_digest(wheel) != entry.get('content_digest'): raise RuntimeError(f'runtime wheel binding mismatch: {logical}')
    with zipfile.ZipFile(wheel) as archive:
        for member in archive.infolist():
            parts = member.filename.split('/'); parts = parts[:-1] if member.is_dir() and parts[-1] == '' else parts
            if not member.filename or not parts or '\\' in member.filename or member.filename.startswith('/') or any(part in ('', '.', '..') for part in parts) or stat.S_ISLNK(member.external_attr >> 16): raise RuntimeError(f'unsafe runtime wheel member: {logical}:{member.filename}')
            if member.is_dir(): continue
            if member.filename in extracted: raise RuntimeError(f'colliding runtime wheel member: {member.filename}')
            extracted.add(member.filename); destination = runtime_site.joinpath(*parts); destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, destination.open('wb') as sink: shutil.copyfileobj(source, sink)
os.environ['PYTHONPATH'] = os.pathsep.join((str(WORKING / 'irisx-trial/runtime'), str(runtime_site)))
os.environ['PYTHONNOUSERSITE'] = '1'
dataset = trial_manifest['metadata']['model_dataset']
model_root = INPUT / 'datasets' / dataset['owner'] / dataset['slug']
if not model_root.is_dir(): raise RuntimeError(f'bound raw model Dataset is missing: {model_root}')
os.environ['IRISX_KAGGLE_INTERNET_DISABLED'] = '1'
runtime_probe = subprocess.run([sys.executable, '-c', 'import transformers; from transformers import Qwen3_5Config; print(transformers.__version__)'], check=True, capture_output=True, text=True)
if runtime_probe.stdout.strip() != '5.8.1': raise RuntimeError(f'unexpected effective transformers runtime: {runtime_probe.stdout.strip()}')
runner = WORKING / 'irisx-trial/runtime/runner.py'
checkpoint_output = WORKING / 'checkpoint-package.zip'
resume = checkpoint_output if checkpoint_output.is_file() else CHECKPOINT_DATASET_ROOT / 'checkpoint-package.zip'
command = [sys.executable, str(runner), '--trial-root', str(WORKING / 'irisx-trial'), '--model-root', str(model_root), '--output', str(WORKING / 'result-package.zip'), '--checkpoint-output', str(checkpoint_output), '--trial-manifest-digest', trial_manifest_digest]
if resume.is_file(): command.extend(['--resume-checkpoint', str(resume)])
subprocess.run(command, check=True)
print('checkpoint:', checkpoint_output)
print('result:', WORKING / 'result-package.zip')
